In [26]:
source ../modules/setup.tcl

set year 2017
set day 23

aoc::get-puzzle $year $day 1
aoc::get-puzzle $year $day 2
set input [string trim [aoc::get-input $year $day]]
jupyter::display "text/markdown" "## Input
```
[string range $input 0 100]...
```";

(cached)



## --- Day 23: Coprocessor Conflagration ---



You decide to head directly to the CPU and fix the printer from there. As you get close, you find an <b>experimental coprocessor</b> doing so much work that the local programs are afraid it will [halt and catch fire](https://en.wikipedia.org/wiki/Halt_and_Catch_Fire). This would cause serious issues for the rest of the computer, so you head in and see what you can do.


The code it's running seems to be a variant of the kind you saw recently on that [tablet](18). The general functionality seems <b>very similar</b>, but some of the instructions are different:


- 
<code>set X Y</code> <b>sets</b> register <code>X</code> to the value of <code>Y</code>.

- 
<code>sub X Y</code> <b>decreases</b> register <code>X</code> by the value of <code>Y</code>.

- 
<code>mul X Y</code> sets register <code>X</code> to the result of <b>multiplying</b> the value contained in register <code>X</code> by the value of <code>Y</code>.

- 
<code>jnz X Y</code> <b>jumps</b> with an offset of the value of <code>Y</code>, but only if the value of <code>X</code> is <b>not zero</b>. (An offset of <code>2</code> skips the next instruction, an offset of <code>-1</code> jumps to the previous instruction, and so on.)


Only the instructions listed above are used. The eight registers here, named <code>a</code> through <code>h</code>, all start at <code>0</code>.




The coprocessor is currently set to some kind of <b>debug mode</b>, which allows for testing, but prevents it from doing any meaningful work.


If you run the program (your puzzle input), <b>how many times is the <code>mul</code> instruction invoked?</b>



(cached)



## --- Part Two ---



Now, it's time to fix the problem.


The <b>debug mode switch</b> is wired directly to register <code>a</code>.  You flip the switch, which makes <b>register <code>a</code> now start at <code>1</code>
</b> when the program is executed.


Immediately, the coprocessor begins to overheat.  Whoever wrote this program obviously didn't choose a very efficient implementation.  You'll need to <b>optimize the program</b> if it has any hope of completing before Santa needs that printer working.


The coprocessor's ultimate goal is to determine the final value left in register <code>h</code> once the program completes. Technically, if it had that... it wouldn't even need to run the program.


After setting register <code>a</code> to <code>1</code>, if the program were to run to completion, <b>what value would be left in register <code>h</code>?</b>



(cached)

## Input
```
set b 81
set c b
jnz a 2
jnz 1 5
mul b 100
sub b -100000
set c b
sub c -17000
set f 1
set d 2
set e 2...
```

In [27]:
set pc 0
set muls 0
set prog [lmap x [split $input \n] {split $x}]

set size [llength $prog]
unset -nocomplain regs
array set regs [list a 0 b 0 c 0 d 0 e 0 f 0 h 0]

proc valof {arg} {
    variable regs
    variable pc
    if {[string is integer $arg]} {
        return $arg
    } {
        return $regs($arg)
    }
    incr pc
}
proc op_set {reg arg} {
    variable regs
    variable pc
    set regs($reg) [valof $arg]
    incr pc
}
proc op_jnz {test offset} {
    variable pc
    if {[valof $test] != 0} {
        incr pc $offset
    } else {
        incr pc
    }

}
proc op_mul {reg arg1} {
    variable regs
    variable muls
    variable pc
    set regs($reg) [* [valof $arg1] $regs($reg)]
    incr pc
    incr muls
}
proc op_sub {reg arg1} {
    variable regs
    variable pc
    set val [- [valof $arg1]]
    incr regs($reg) $val
    incr pc
}

proc part1 {} {
    set steps 0
    variable input
    variable prog
    variable pc
    variable muls
    while 1 {
        set cmd [lindex $prog $pc]
        set args [lassign $cmd op]
        if {$op eq {}} {
            break
        }
        op_$op {*}$args
        incr steps
        }
        return $muls
}

In [28]:
proc naive {a} {
    # some rewriting of the assembly code gives this
    set b 81                                            
    set c $b
    set h 0
    if {$a !=0 } {                                      
        set b [expr {$b*100 + 100000}]                                         
        set c $b                                          
        incr c 17000                                        
    }
#     puts "$b:$c"
    puts [expr {($c-$b)/17}]
     for {set n $b} {$n <= $c} {incr n 17}  { 
        set f 1                                        
        for {set d 2} {$d <= $n} {incr d} { 
            for {set e 2} {$e <= $n} {incr e} {                      
                if {$d*$e == $n } {
                    set f 0
                }
            }
        }
        if {$f == 0} {incr h}
    }
    return $h
}
# naive 1 -> this will not end anytime soon

### On fire? How come?

Looking at this closely it loops for all numbers `n` `$b` <= `$n` <= `$c` where `n` increases with 17. It then increases register `h` if the number was not prime.

For my input `$b` == 108100 and `$c` ==  125100 this are `($c-$b)/17+1 == 1001` numbers to check.

It does this by calculating if the number is equal to any of the products `1..n * 1..n` without stopping the check if the non-primeness is alreay established.  Hence the fire risk.

So easier to just check the primeness directly as shown in the `smart` solution below.



In [29]:
proc primetest n {
    set max [expr wide(sqrt($n))]
    if {$n%2==0} {return false}
    for {set i 3} {$i<=$max} {incr i 2} {
        if {$n%$i==0} {return false}
    }
    return true
}

proc smart {a} {
    set b 81                                            
    set c $b
    set h 0
    if {$a !=0 } {                                      
        set b [expr {$b*100 + 100000}]                                         
        set c $b                                          
        incr c 17000                                        
    }
#     puts "$b:$c"
     for {set n $b} {$n <= $c} {incr n 17}  { 
        if {![primetest $n]} {incr h}
       

    }
    return $h
}
# smart 1

In [30]:
aoc::parts {
    set input [yield]
    yield [part1]
    yield [smart 1]
}
aoc::results 

Part1	6241 (69429 us)
Part2	909 (3598 us)


6241 909